# AI Smart Recruitment System — Deep Learning Pipeline (BERT & LLM)

This notebook walks through the **core deep-learning pipeline** behind the AI Smart Recruitment System:

1. Resume parsing & information extraction
2. Semantic skill matching with **Sentence-BERT**
3. AI-powered ATS score
4. Skill-gap analysis
5. Candidate ranking
6. AI resume summary (embedding-centrality)
7. AI-generated HR & technical interview questions
8. Exporting results (CSV / JSON) for use in the Flask app

> Run this top-to-bottom in Google Colab: **Runtime → Run all**. No GPU required (Sentence-BERT's MiniLM model runs fine on CPU), but Colab will use a GPU automatically if available.


## 1. Install dependencies

In [ ]:
!pip install -q sentence-transformers PyMuPDF pandas matplotlib


## 2. Imports

In [ ]:
import re
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer, util

pd.set_option("display.max_colwidth", 120)


## 3. Skill taxonomy

The same taxonomy used by the Flask app (`utils/skills_data.py`) — grouped by category so
we can do category-aware skill-gap analysis and pick relevant technical interview questions.


In [ ]:
SKILL_TAXONOMY = {
    "Programming Languages": ["python", "java", "c++", "c", "javascript", "typescript", "go", "rust",
                               "r", "sql", "kotlin", "swift", "php", "matlab", "scala"],
    "Web Development": ["html", "css", "react", "angular", "vue", "flask", "django", "fastapi",
                         "node.js", "express", "next.js", "rest api", "graphql", "bootstrap", "tailwind"],
    "Data & ML": ["machine learning", "deep learning", "nlp", "computer vision", "pytorch", "tensorflow",
                  "keras", "scikit-learn", "pandas", "numpy", "opencv", "bert", "transformers", "llm",
                  "generative ai", "mlops", "data analysis", "data visualization", "statistics",
                  "feature engineering"],
    "Databases": ["mysql", "postgresql", "mongodb", "sqlite", "redis", "oracle", "firebase",
                  "cassandra", "dynamodb"],
    "Cloud & DevOps": ["aws", "azure", "gcp", "docker", "kubernetes", "ci/cd", "jenkins", "terraform",
                        "linux", "git", "github actions", "nginx"],
    "Core CS": ["data structures", "algorithms", "oop", "operating systems", "computer networks",
                "system design", "dbms", "software engineering"],
}

SKILL_TO_CATEGORY = {s: cat for cat, skills in SKILL_TAXONOMY.items() for s in skills}
ALL_SKILLS = sorted(SKILL_TO_CATEGORY.keys(), key=len, reverse=True)
print(f"{len(ALL_SKILLS)} skills across {len(SKILL_TAXONOMY)} categories")


## 4. Sample data

For a self-contained demo we use synthetic resumes + a job description. **To use your own
resumes**, uncomment the upload cell below (works with PDF/DOCX/TXT via PyMuPDF / python-docx),
or paste resume text directly into `raw_resumes`.


In [ ]:
# Optional: upload real resume files in Colab
# from google.colab import files
# uploaded = files.upload()   # then extract text with fitz.open(filename) as shown in section 5b

job_description = '''
We are hiring a Machine Learning Engineer to build NLP and recommendation systems.
Required skills: Python, Machine Learning, Deep Learning, PyTorch, BERT, NLP, SQL, Docker, AWS.
2+ years of experience preferred. Familiarity with Transformers and MLOps is a plus.
'''

raw_resumes = {
    "Aditi Rao": '''
        Aditi Rao | aditi.rao@email.com | +91 90000 11111
        Machine learning engineer with 3 years experience building NLP pipelines using
        Python, PyTorch, and BERT-based transformers. Deployed models on AWS with Docker.
        Skilled in SQL, Scikit-learn, Pandas, Numpy, and MLOps practices.
        B.Tech in Computer Science.
    ''',
    "Karthik Iyer": '''
        Karthik Iyer | karthik.iyer@email.com | +91 90000 22222
        Frontend developer with 4 years experience in React, JavaScript, HTML, CSS, and Node.js.
        Some exposure to Python scripting and REST APIs. No formal ML background.
    ''',
    "Meera Nair": '''
        Meera Nair | meera.nair@email.com | +91 90000 33333
        Data scientist with 1.5 years experience in Machine Learning and Deep Learning.
        Proficient in Python, TensorFlow, Keras, NLP, and SQL. Currently learning Docker and AWS.
        M.Sc in Data Science.
    ''',
}

print(f"Loaded {len(raw_resumes)} sample resumes.")


## 5. Resume parsing & information extraction

In the Flask app this reads PDFs/DOCX via **PyMuPDF** (`utils/parser.py`). Here we parse the
raw text directly since our sample resumes are plain strings — the extraction logic
(regex + skill-dictionary matching) is identical.


In [ ]:
EMAIL_RE = re.compile(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}")
PHONE_RE = re.compile(r"(\+?\d{1,3}[-.\s]?)?\(?\d{3,5}\)?[-.\s]?\d{3,4}[-.\s]?\d{3,4}")
EXPERIENCE_RE = re.compile(r"(\d+(?:\.\d+)?)\s*\+?\s*(?:years|yrs|year)\s*(?:of)?\s*experience", re.I)

def extract_skills(text: str) -> list:
    lowered = text.lower()
    found = set()
    for skill in ALL_SKILLS:
        pattern = r"(?<![a-zA-Z0-9])" + re.escape(skill) + r"(?![a-zA-Z0-9])"
        if re.search(pattern, lowered):
            found.add(skill)
    return sorted(found)

def estimate_experience_years(text: str) -> float:
    matches = EXPERIENCE_RE.findall(text)
    return max((float(m) for m in matches), default=0.0)

def parse_resume_text(name: str, text: str) -> dict:
    email = EMAIL_RE.search(text)
    phone = PHONE_RE.search(text)
    skills = extract_skills(text)
    return {
        "name": name,
        "email": email.group() if email else None,
        "phone": phone.group().strip() if phone else None,
        "skills": skills,
        "experience_years": estimate_experience_years(text),
        "raw_text": text.strip(),
    }

parsed_candidates = [parse_resume_text(name, text) for name, text in raw_resumes.items()]
pd.DataFrame(parsed_candidates)[["name", "email", "experience_years", "skills"]]


## 6. Load Sentence-BERT

`all-MiniLM-L6-v2` — a distilled Sentence-BERT model that maps text to a 384-dimensional
embedding space where cosine similarity reflects **semantic** similarity, not just keyword
overlap. This is the deep-learning core of the semantic skill-matching module.


In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")
print("Model loaded:", model)


## 7. Semantic similarity — resume vs job description

In [ ]:
def semantic_similarity(text_a: str, text_b: str) -> float:
    emb = model.encode([text_a, text_b], convert_to_tensor=True, normalize_embeddings=True)
    score = util.cos_sim(emb[0], emb[1]).item()
    return max(0.0, min(1.0, (score + 1) / 2 if score < 0 else score))

for c in parsed_candidates:
    c["semantic_score"] = round(semantic_similarity(c["raw_text"], job_description) * 100, 1)

pd.DataFrame(parsed_candidates)[["name", "semantic_score"]].sort_values("semantic_score", ascending=False)


## 8. AI-powered ATS score

Weighted blend: **40% semantic similarity + 40% keyword/skill overlap + 20% experience match**.
This mirrors `utils/matcher.py::compute_ats_score` in the Flask app.


In [ ]:
job_skills = extract_skills(job_description)
required_experience = 2.0
print("Required skills detected from JD:", job_skills)

def skill_overlap_score(resume_skills, jd_skills):
    if not jd_skills:
        return 0.0
    matched = set(resume_skills) & set(jd_skills)
    return len(matched) / len(set(jd_skills))

def verdict(score):
    if score >= 80: return "Excellent Match"
    if score >= 65: return "Strong Match"
    if score >= 45: return "Moderate Match"
    return "Weak Match"

for c in parsed_candidates:
    kw_score = skill_overlap_score(c["skills"], job_skills)
    exp_score = min(1.0, c["experience_years"] / required_experience) if required_experience > 0 else 1.0
    sem_score = c["semantic_score"] / 100

    final = 0.40 * sem_score + 0.40 * kw_score + 0.20 * exp_score
    c["keyword_score"] = round(kw_score * 100, 1)
    c["experience_score"] = round(exp_score * 100, 1)
    c["ats_score"] = round(final * 100, 1)
    c["verdict"] = verdict(c["ats_score"])

pd.DataFrame(parsed_candidates)[["name", "semantic_score", "keyword_score", "experience_score", "ats_score", "verdict"]]


## 9. Skill-gap analysis

In [ ]:
def skill_gap(resume_skills, jd_skills):
    resume_set, jd_set = set(resume_skills), set(jd_skills)
    matched = sorted(resume_set & jd_set)
    missing = sorted(jd_set - resume_set)
    missing_by_category = {}
    for s in missing:
        cat = SKILL_TO_CATEGORY.get(s, "Other")
        missing_by_category.setdefault(cat, []).append(s)
    return {"matched": matched, "missing": missing, "missing_by_category": missing_by_category}

for c in parsed_candidates:
    gap = skill_gap(c["skills"], job_skills)
    c["matched_skills"] = gap["matched"]
    c["missing_skills"] = gap["missing"]
    print(f"--- {c['name']} ---")
    print("  Matched:", gap["matched"])
    print("  Missing:", gap["missing"])


## 10. Candidate ranking

In [ ]:
ranked = sorted(parsed_candidates, key=lambda c: c["ats_score"], reverse=True)
for i, c in enumerate(ranked, start=1):
    c["rank"] = i

rank_df = pd.DataFrame(ranked)[["rank", "name", "ats_score", "verdict"]]
rank_df


In [ ]:
plt.figure(figsize=(7, 4))
names = [c["name"] for c in ranked]
scores = [c["ats_score"] for c in ranked]
colors = ["#e4a93b" if i == 0 else "#5468f5" for i in range(len(ranked))]
plt.bar(names, scores, color=colors)
plt.axhline(65, color="gray", linestyle="--", linewidth=1, label="Strong-match threshold")
plt.ylabel("ATS Score")
plt.title("Candidate ranking — ATS score")
plt.ylim(0, 100)
plt.legend()
plt.tight_layout()
plt.show()


## 11. AI resume summary (embedding-centrality)

An offline, no-API-key summarizer: split the resume into sentences, embed each with
Sentence-BERT, then pick the sentences closest to the document's mean embedding — i.e.
the most *representative* sentences. This is the same technique used in `utils/summarizer.py`.


In [ ]:
def split_sentences(text):
    text = re.sub(r"\s+", " ", text).strip()
    sentences = re.split(r"(?<=[.!?])\s+", text)
    return [s.strip() for s in sentences if 6 <= len(s.split()) <= 60]

def summarize_resume(resume_text, name, top_k=3):
    sentences = split_sentences(resume_text)
    if len(sentences) <= top_k:
        return " ".join(sentences)
    embeddings = model.encode(sentences, convert_to_tensor=True, normalize_embeddings=True)
    doc_embedding = embeddings.mean(dim=0)
    scores = util.cos_sim(doc_embedding, embeddings)[0]
    top_idx = sorted(range(len(sentences)), key=lambda i: scores[i], reverse=True)[:top_k]
    top_idx.sort()
    return f"{name} — " + " ".join(sentences[i] for i in top_idx)

for c in ranked:
    c["summary"] = summarize_resume(c["raw_text"], c["name"])
    print(f"\n{c['name']}:\n{c['summary']}")


## 12. AI-generated HR & technical interview questions

A curated, skill-grounded question bank (same as `utils/questions.py`) avoids hallucinated
technical questions. Swap in a hosted LLM call here if you want fully generative questions
(see the Flask app's `utils/llm.py` for an OpenAI/Groq hook).


In [ ]:
TECHNICAL_QUESTION_BANK = {
    "python": ["Explain the difference between a list and a tuple in Python.",
               "What are Python decorators and when would you use one?"],
    "sql": ["What's the difference between INNER JOIN and LEFT JOIN?",
            "How would you optimize a slow-running SQL query?"],
    "machine learning": ["How would you handle an imbalanced dataset?",
                          "Explain the bias-variance tradeoff."],
    "deep learning": ["What's the difference between a CNN and an RNN?",
                       "How do you prevent a neural network from overfitting?"],
    "bert": ["How does BERT's attention mechanism differ from earlier NLP models?",
              "What's the difference between BERT and Sentence-BERT?"],
    "docker": ["What's the difference between a Docker image and a container?"],
    "aws": ["What's the difference between an EC2 instance and a Lambda function?"],
}

HR_QUESTIONS = [
    "Tell me about yourself and what draws you to this role.",
    "Describe a challenging project you worked on and how you handled obstacles.",
    "How do you prioritize tasks when working under tight deadlines?",
]

def generate_questions(matched_skills, n_technical=5):
    technical = []
    for skill in matched_skills:
        for q in TECHNICAL_QUESTION_BANK.get(skill, []):
            if q not in technical:
                technical.append(q)
        if len(technical) >= n_technical:
            break
    return technical[:n_technical], HR_QUESTIONS

for c in ranked:
    tech_qs, hr_qs = generate_questions(c["matched_skills"])
    c["technical_questions"] = tech_qs
    c["hr_questions"] = hr_qs
    print(f"\n=== {c['name']} — Technical Questions ===")
    for q in tech_qs:
        print(" -", q)


## 13. Export results

Export ranked candidates (with scores, skill gaps, summaries, and questions) as JSON/CSV —
useful for feeding into the Flask app's database, a spreadsheet, or a BI tool.


In [ ]:
export_df = pd.DataFrame(ranked)[[
    "rank", "name", "email", "ats_score", "semantic_score", "keyword_score",
    "experience_score", "verdict", "matched_skills", "missing_skills"
]]
export_df.to_csv("ranked_candidates.csv", index=False)

with open("ranked_candidates.json", "w") as f:
    json.dump(ranked, f, indent=2)

print("Saved ranked_candidates.csv and ranked_candidates.json")
export_df


In [ ]:
# In Colab, download the exported files:
# from google.colab import files
# files.download("ranked_candidates.csv")
# files.download("ranked_candidates.json")


## 14. Next steps

- Swap the synthetic resumes for real ones (upload PDFs and parse with `PyMuPDF`, exactly
  as `utils/parser.py` does in the Flask app).
- Fine-tune `all-MiniLM-L6-v2` on a labeled resume-JD relevance dataset if you want the
  semantic score to be even sharper for your domain.
- Plug a hosted LLM (OpenAI / Groq / Gemini) into the question generator and summarizer for
  fully generative output — see `utils/llm.py` in the Flask project.
- The Flask app (`app.py`) reuses every function shown here — this notebook is meant to be
  a sandbox for experimenting with the model before/alongside the deployed app.
